In [1]:
import os

import pandas as pd
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
load_dotenv()


True

In [2]:
api_key = os.getenv("OLLAMA_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")

headers = {
   "Authorization": f"Bearer {api_key}"
}


In [3]:
llm_ollama = ChatOllama(
   # base_url="https://ollama.com", # 원격 서버 주소
   model="gemma4:31b-mlx",
   # client_kwargs={"headers": headers},
   temperature=0.2,
   reasoning=True
)

In [4]:
llm_openai = ChatOpenAI(
    model='gpt-5-nano',
    temperature=0.2,
    reasoning_effort="medium",
    api_key=openai_api_key
)

In [6]:
subway_2016 = pd.read_csv("../data/subway/subway.csv")

In [9]:
target_subway = ['강남', '사당', '잠실', '서울역', '홍대입구']
subway_2016_target = subway_2016[subway_2016['지하철역명'].isin(target_subway)].copy()

In [14]:
subway_2016_target.날짜 = pd.to_datetime(subway_2016_target.날짜)

In [34]:
summary_data = subway_2016_target[(subway_2016_target.날짜 >= '2016-07-01') & (subway_2016_target.날짜 <= '2016-07-31')].to_json()

/var/folders/2v/b65115v14cvgvjztf82wr9x80000gn/T/ipykernel_8837/2259198472.py:1: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  summary_data = subway_2016_target[(subway_2016_target.날짜 >= '2016-07-01') & (subway_2016_target.날짜 <= '2016-07-31')].to_json()


In [24]:
prompt = """
당신은 소상공인을 위한 컨설턴트입니다.

사용자가 지하철 역에서 김밥을 팔려고 합니다. 어느 역에서 팔아야 할지 아래 데이터를 기반으로 대답해주세요.

{data}
\n\n\n

{question}
"""

In [35]:
subway_prompt = PromptTemplate(
    template=prompt,
    input_variables=['question'],
    partial_variables={'data': summary_data}
)

In [51]:
chain = subway_prompt | llm_openai | StrOutputParser()

In [52]:
chain.invoke({'question': "어디 가서 김밥을 팔아야 내가 돈을 많이 벌까?"})

'요약부터 말하자면, 지하철역 중에서 김밥 판매로 가장 돈을 많이 벌 가능성이 높은 곳은 강남역입니다.\n\n이유 및 근거(데이터 기반 요약)\n- 데이터 세트는 각 역에 대해 하루 24시간대의 예상 매출값을 hour 단위로 제공합니다. 총합이 가장 큰 역이 가장 수익성이 좋은 곳이고, 대도시의 핵심 허브인 강남역 쪽의 합계가 다른 역들에 비해 월등히 높습니다.\n- 강남역은 출구 너무 많고 유동인구가 많아 점심시간대와 퇴근 시간대에 매출이 크게 뛸 가능성이 큽니다. 데이터에서 00~01부터 23~24까지의 여러 시간대에서 강남역 코드의 매출 합계가 다른 역들에 비해 높게 나타납니다.\n- 따라서 초기 진입은 강남역에서 시작하는 것이 수익을 극대화하는 데 가장 합리적일 가능성이 큽니다.\n\n다음도 고려할 만한 대안 (2~3위 후보)\n- 잠실역: 대형 쇼핑몰/롯데타워 인근 등으로 유동인구가 많아 매출 잠재력 높음.\n- 역삼역 또는 홍대입구역: 업무/유동인구가 몰리는 구역으로 비교적 높은 매출 가능성이 있습니다.\n- 이외의 대형 허브역들도 분위기에 따라 성과 차이가 크므로, 상권 특성(출구 방향, 인근 상가 밀도, 보안/허가 여부 등)도 함께 점검 필요.\n\n실행 제안(실제 숫자 확인을 원하시면 제가 계산해 드리겠습니다)\n- 전체 역의 24시간 매출 합계를 구해 1위 ~ 5위 역을 뽑고, 각 역의 일일 총매출 예상치와 시간대별 피크 시간을 제시해 드리겠습니다.\n- 최적 선택 시점 시간대(예: 점심 11–13시, 퇴근 17–19시)와 권장 포장/재고 규모도 함께 제안드릴 수 있습니다.\n- 장기적으로는 통행량 변화에 따른 시즌성도 반영해 주간/월간 계획으로 확장하는 것을 권합니다.\n\n다음 단계로 원하시면:\n- 제가 데이터를 바탕으로 실제 합산을 해서 1위 역(강남역)과 2위~5위 역의 정확한 일일 매출 추정치를 표로 드리겠습니다.\n- 또한 각 역별로 피크 시간대와 추천 판매 방식(포장 위주 vs 매장 내 즉시판매), 가격대(예: 김밥 3,000원

In [5]:
from langchain_core.runnables import RunnablePassthrough

In [6]:
with open("../data/소설.txt", "r") as f:
    novel = f.read()

In [13]:
import re

from langchain_postgres import PGEngine, PGVectorStore
from langchain_core.documents import Document

In [14]:
p = re.compile(r"(\d+화\.\s*[^\n]+)")

In [16]:
p.findall("1화. 간판")

['1화. 간판']

In [17]:
parts = re.split(r"(\d+화\.\s*[^\n]+)", novel)

In [20]:
parts[:3]

['같은 골목\n\n',
 '1화. 간판이 하나 더 생겼다',
 '\n문 닫기 전, 나는 늘 가게를 한 바퀴 돌았다. 싱크대 수건을 반듯하게 펴 걸고, 고무장갑을 뒤집어 물기를 털어 말리고, 가스 밸브를 한 번 더 확인했다. 다 알고 있는 일인데도 꼭 손으로 만져봐야 마음이 놓였다. 스무 평 남짓한 가게 안에는 하루 종일 끓여낸 육수 냄새와 김치의 시큼한 열기가 가라앉아 있었다. 마지막 손님이 나가고 난 뒤의 그 적막이 나는 싫지 않았다. 오히려 그 시간이 하루 중 가장 내 것이었다.\n문을 잠그고 골목으로 나왔을 때, 먼저 눈에 들어온 건 맞은편의 불빛이었다. 며칠 전까지 비어 있던 가게였다. 부동산 종이가 붙어 있던 유리창이 환하게 밝아져 있었다. 누가 들어왔나 싶어 무심코 고개를 들었다가 그대로 발이 멈췄다.\n간판에 적힌 글자를 읽는 데 몇 초가 걸렸다.\n‘집밥김치찜.’\n내 가게 상호는 아니었다. 하지만 너무 익숙한 말들이었다. 김치찜. 집밥. 단일 메뉴. 누구나 생각할 수 있는 조합인데도, 나는 순간 숨이 턱 막혔다. 유리 앞 메뉴판으로 시선이 내려갔다. 2인, 3인, 4인. 가격도 비슷했다. 아주 조금, 정말 조금 더 쌌다.\n문이 열리고 누군가 나왔다. 허리에 앞치마를 두른 채 휴대폰을 들고 있던 사람이 나를 보고 잠깐 멈췄다.\n동생이었다.\n“왔어?”\n평소랑 똑같은 목소리였다. 마트에서 우연히 마주친 것처럼, 아무 일도 아니라는 얼굴이었다.\n나는 대답을 못 했다. 눈앞에 있는 게 간판인지, 동생인지, 메뉴판인지 잘 구분이 안 됐다. 골목 끝 치킨집에서 흘러나오는 라디오 소리와 멀리서 들리는 오토바이 소리만 괜히 또렷했다.\n“여기 뭐야.”\n겨우 나온 말이 그것뿐이었다.\n동생이 짧게 대답했다.\n“가게.”\n“보면 몰라서 묻는 줄 알아?”\n내 목소리는 생각보다 낮았다. 화를 내는 것도, 따지는 것도 아닌 애매한 톤이었다.\n동생은 시선을 한번 피했다가 다시 나를 봤다. “말하려고 했어.”\n“언제.”\n“정리되면.”\n나는 메뉴판을 

In [21]:
Document(page_content=parts[2])

Document(metadata={}, page_content='\n문 닫기 전, 나는 늘 가게를 한 바퀴 돌았다. 싱크대 수건을 반듯하게 펴 걸고, 고무장갑을 뒤집어 물기를 털어 말리고, 가스 밸브를 한 번 더 확인했다. 다 알고 있는 일인데도 꼭 손으로 만져봐야 마음이 놓였다. 스무 평 남짓한 가게 안에는 하루 종일 끓여낸 육수 냄새와 김치의 시큼한 열기가 가라앉아 있었다. 마지막 손님이 나가고 난 뒤의 그 적막이 나는 싫지 않았다. 오히려 그 시간이 하루 중 가장 내 것이었다.\n문을 잠그고 골목으로 나왔을 때, 먼저 눈에 들어온 건 맞은편의 불빛이었다. 며칠 전까지 비어 있던 가게였다. 부동산 종이가 붙어 있던 유리창이 환하게 밝아져 있었다. 누가 들어왔나 싶어 무심코 고개를 들었다가 그대로 발이 멈췄다.\n간판에 적힌 글자를 읽는 데 몇 초가 걸렸다.\n‘집밥김치찜.’\n내 가게 상호는 아니었다. 하지만 너무 익숙한 말들이었다. 김치찜. 집밥. 단일 메뉴. 누구나 생각할 수 있는 조합인데도, 나는 순간 숨이 턱 막혔다. 유리 앞 메뉴판으로 시선이 내려갔다. 2인, 3인, 4인. 가격도 비슷했다. 아주 조금, 정말 조금 더 쌌다.\n문이 열리고 누군가 나왔다. 허리에 앞치마를 두른 채 휴대폰을 들고 있던 사람이 나를 보고 잠깐 멈췄다.\n동생이었다.\n“왔어?”\n평소랑 똑같은 목소리였다. 마트에서 우연히 마주친 것처럼, 아무 일도 아니라는 얼굴이었다.\n나는 대답을 못 했다. 눈앞에 있는 게 간판인지, 동생인지, 메뉴판인지 잘 구분이 안 됐다. 골목 끝 치킨집에서 흘러나오는 라디오 소리와 멀리서 들리는 오토바이 소리만 괜히 또렷했다.\n“여기 뭐야.”\n겨우 나온 말이 그것뿐이었다.\n동생이 짧게 대답했다.\n“가게.”\n“보면 몰라서 묻는 줄 알아?”\n내 목소리는 생각보다 낮았다. 화를 내는 것도, 따지는 것도 아닌 애매한 톤이었다.\n동생은 시선을 한번 피했다가 다시 나를 봤다. “말하려고 했어.”\n“언제.”\n“정리되면.”\n나는 메뉴판을 가

In [12]:
print(novel)

같은 골목

1화. 간판이 하나 더 생겼다
문 닫기 전, 나는 늘 가게를 한 바퀴 돌았다. 싱크대 수건을 반듯하게 펴 걸고, 고무장갑을 뒤집어 물기를 털어 말리고, 가스 밸브를 한 번 더 확인했다. 다 알고 있는 일인데도 꼭 손으로 만져봐야 마음이 놓였다. 스무 평 남짓한 가게 안에는 하루 종일 끓여낸 육수 냄새와 김치의 시큼한 열기가 가라앉아 있었다. 마지막 손님이 나가고 난 뒤의 그 적막이 나는 싫지 않았다. 오히려 그 시간이 하루 중 가장 내 것이었다.
문을 잠그고 골목으로 나왔을 때, 먼저 눈에 들어온 건 맞은편의 불빛이었다. 며칠 전까지 비어 있던 가게였다. 부동산 종이가 붙어 있던 유리창이 환하게 밝아져 있었다. 누가 들어왔나 싶어 무심코 고개를 들었다가 그대로 발이 멈췄다.
간판에 적힌 글자를 읽는 데 몇 초가 걸렸다.
‘집밥김치찜.’
내 가게 상호는 아니었다. 하지만 너무 익숙한 말들이었다. 김치찜. 집밥. 단일 메뉴. 누구나 생각할 수 있는 조합인데도, 나는 순간 숨이 턱 막혔다. 유리 앞 메뉴판으로 시선이 내려갔다. 2인, 3인, 4인. 가격도 비슷했다. 아주 조금, 정말 조금 더 쌌다.
문이 열리고 누군가 나왔다. 허리에 앞치마를 두른 채 휴대폰을 들고 있던 사람이 나를 보고 잠깐 멈췄다.
동생이었다.
“왔어?”
평소랑 똑같은 목소리였다. 마트에서 우연히 마주친 것처럼, 아무 일도 아니라는 얼굴이었다.
나는 대답을 못 했다. 눈앞에 있는 게 간판인지, 동생인지, 메뉴판인지 잘 구분이 안 됐다. 골목 끝 치킨집에서 흘러나오는 라디오 소리와 멀리서 들리는 오토바이 소리만 괜히 또렷했다.
“여기 뭐야.”
겨우 나온 말이 그것뿐이었다.
동생이 짧게 대답했다.
“가게.”
“보면 몰라서 묻는 줄 알아?”
내 목소리는 생각보다 낮았다. 화를 내는 것도, 따지는 것도 아닌 애매한 톤이었다.
동생은 시선을 한번 피했다가 다시 나를 봤다. “말하려고 했어.”
“언제.”
“정리되면.”
나는 메뉴판을 가리켰다. “이게 정리된 거 아니야?”
동생은 대답하지 않았다

In [22]:
from langchain_ollama import OllamaEmbeddings

embedding = OllamaEmbeddings(model="embeddinggemma:300m")

In [23]:
len(embedding.embed_query("대한민국"))

768

In [24]:
DB_USER = os.getenv("DB_USER", "langchain")
DB_PASSWORD = os.getenv("DB_PASSWORD", "langchain")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "langchain")


In [25]:
CONNECTION_STRING = (
    f"postgresql+psycopg://"
    f"{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)


In [26]:
engine = PGEngine.from_connection_string(
    url=CONNECTION_STRING,
)


In [36]:
engine.init_vectorstore_table(
    table_name='novel',
    vector_size=768
)

In [37]:
vector_store = PGVectorStore.create_sync(
    engine=engine,
    table_name="novel",
    embedding_service=embedding,
)

In [38]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [40]:
splitter = RecursiveCharacterTextSplitter(          
         # RecursiveCharacterTextSplitter 객체를 생성한다.
    chunk_size=50,                                        
          # 청크 목표 크기를 50자로 설정한다.
    chunk_overlap=10                                            
         # 청크 사이에 10자 정도 겹치게 설정한다.
)

In [42]:
from typing import List

def split_by_episode(raw_text: str, source_name: str) -> List[Document]:
    parts = re.split(r"(\d+화\.\s*[^\n]+)", raw_text)
    documents = []
    for i in range(1, len(parts), 2):
        episode_title = parts[i].strip()  # 예: "1화. 간판이 하나 더 생겼다"
        episode_body = parts[i + 1].strip() if i + 1 < len(parts) else ""

        # "1화"에서 숫자 부분만 추출합니다.
        match = re.match(r"(\d+)화\.", episode_title)
        episode_number = int(match.group(1)) if match else None

        # 제목과 본문을 다시 합쳐 하나의 에피소드 문서로 만듭니다.
        full_text = f"{episode_title}\n{episode_body}"

        documents.append(
            Document(
                page_content=full_text,
                metadata={
                    "source": source_name,          # 어떤 파일에서 왔는지
                    "episode": episode_number,      # 몇 화인지
                    "episode_title": episode_title, # 에피소드 제목
                },
            )
        )
    return documents


In [43]:
episode_docs = split_by_episode(novel, source_name="소설")


In [44]:
len(episode_docs)

10

In [45]:
splitter = RecursiveCharacterTextSplitter(
    separators=['\n\n', '\n', '. ', ' ', ""],
    chunk_size=700,
    chunk_overlap = 120,
    length_function=len
)


In [46]:
chunked_docs = splitter.split_documents(episode_docs)

In [47]:
len(chunked_docs)

22

In [48]:
vector_store.add_documents(
    documents=chunked_docs,
)


['3905bac0-ebe3-4a2f-9d5a-4feb8944b1a7',
 'd4a6cb15-932e-4a37-9b59-8ed415e77d7a',
 'f570c1c5-e576-4db3-9542-537b998b7ab8',
 '32e74ee5-172f-4663-b08e-5296c8dd2084',
 'dedb61b3-9744-42d0-83d8-8f8b2cc25041',
 '5c8c42b5-9dd6-44da-8db8-7cd4f103ac35',
 'b372d149-ca9a-4887-9627-29be3d5fdfcf',
 'bfc82e67-f93b-4635-b866-80ef382aed95',
 'd90802e5-7c47-437b-a9e8-ffce69c0cfef',
 '69a9e3fb-f7ae-495d-bedc-a7cecab2aaa7',
 'b9c1127a-754b-4149-9b80-7244de5b9cbb',
 '8ad7745c-6463-4149-abfe-b55a8d35d2da',
 'b24a1c04-20bb-47b8-adf9-a0f3c07c1e7c',
 '41c0d47f-13eb-471f-a544-b562d6b5d7a4',
 '5db39f7a-d69b-4314-be6b-3994580a0611',
 'c93f2b71-b3fb-4417-abf2-29e1fce5e599',
 'b412abc8-d47d-4fc7-ae03-c10bb27b1fa1',
 '2943a1b5-df25-46c5-88b1-137bf1e14949',
 '366ec976-f18b-4953-b1bf-3ab28f8f269a',
 'cd44be47-5950-4be6-807e-961e7a4ea7f4',
 '69406371-31e1-475e-8ae3-d75af3d4f029',
 'be518c6a-2c5d-444e-b080-acb12be0dbca']

In [62]:
retriever = vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

In [63]:
retriever.invoke("식당의 메뉴는 무엇인가?")

[Document(id='d4a6cb15-932e-4a37-9b59-8ed415e77d7a', metadata={'source': '소설', 'episode': 1, 'episode_title': '1화. 간판이 하나 더 생겼다'}, page_content='동생이었다.\n“왔어?”\n평소랑 똑같은 목소리였다. 마트에서 우연히 마주친 것처럼, 아무 일도 아니라는 얼굴이었다.\n나는 대답을 못 했다. 눈앞에 있는 게 간판인지, 동생인지, 메뉴판인지 잘 구분이 안 됐다. 골목 끝 치킨집에서 흘러나오는 라디오 소리와 멀리서 들리는 오토바이 소리만 괜히 또렷했다.\n“여기 뭐야.”\n겨우 나온 말이 그것뿐이었다.\n동생이 짧게 대답했다.\n“가게.”\n“보면 몰라서 묻는 줄 알아?”\n내 목소리는 생각보다 낮았다. 화를 내는 것도, 따지는 것도 아닌 애매한 톤이었다.\n동생은 시선을 한번 피했다가 다시 나를 봤다. “말하려고 했어.”\n“언제.”\n“정리되면.”\n나는 메뉴판을 가리켰다. “이게 정리된 거 아니야?”\n동생은 대답하지 않았다. 그 침묵이 괜히 더 솔직했다. 이미 다 끝난 뒤였다는 뜻이니까.\n나는 유리 안쪽을 들여다봤다. 테이블은 여섯 개, 벽에는 원목 무늬 시트지가 붙어 있었고, 오픈 축하 화분이 두 개 놓여 있었다. 손님을 받을 준비가 끝난 가게였다. 시험 삼아 해보는 수준이 아니었다. 마음이 불편해졌다기보다, 몸속 어딘가가 천천히 식는 느낌이 들었다.\n“왜 하필 여기야.”\n그제야 동생이 입을 열었다. “이 동네가 익숙하잖아.”\n익숙하다는 말에 짧게 웃음이 났다. 익숙한 건 동네만이 아니었을 것이다. 재료 거래처도, 배달 반경도, 저녁 피크 시간대도. 그걸 다 알려준 사람이 나였다.'),
 Document(id='3905bac0-ebe3-4a2f-9d5a-4feb8944b1a7', metadata={'source': '소설', 'episode': 1, 'episode_title': '1화. 간판이 하나 더 생겼다'}, page_content=

In [59]:
from langchain_core.prompts import ChatPromptTemplate


prompt = ChatPromptTemplate.from_template(
    """
너는 한국어 소설 분석용 RAG 어시스턴트다.

반드시 아래 규칙을 지켜라.
1. 답변은 무조건 한국어로 작성한다.
2. 제공된 컨텍스트 범위 안에서만 답한다.
3. 컨텍스트에 없는 내용은 추측하지 말고 "문서에서 확인되지 않습니다."라고 답한다.
4. 감정 변화, 사건 흐름, 장면의 의미를 또렷하게 설명한다.
5. 가능하면 마지막에 근거가 된 episode 번호를 정리한다.

질문:
{input}

컨텍스트:
{context}
"""
)

In [60]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

document_chain = create_stuff_documents_chain(llm_openai, prompt)

In [64]:
rag_chain = create_retrieval_chain(retriever, document_chain)

In [67]:
result = rag_chain.invoke({'input' : "식당의 메뉴는 무엇인가?"})

In [71]:
print(result['answer'])

- 답변: 식당의 메뉴는 단일 메뉴로, '집밥김치찜'이다. 간판에 적힌 글자가 바로 "집밥김치찜"이며, 이 가게의 메뉴가 단 한 가지임을 "단일 메뉴"라는 표현이 확인된다.

- 장면 및 감정 변화 해설:
  - 처음에 주인공은 이 상황을 혼란스럽고 불안하게 느낀다. 간판/메뉴판/동생이 한꺼번에 섞여 보이며 현실과 기억이 교차하는 불안한 분위기가 형성된다.
  - 간판에 적힌 "집밥김치찜"이라는 익숙한 문구를 보자마자 과거에 익숙했던 단어들이 떠오르고, 현재의 낯섦 속에서도 뿌리 깊은 친숙함이 느껴진다. "익숙하다는 말"이 주인공의 감정에 작은 웃음을 남기지만 동시에 마음을 뒤흔든다.
  - "단일 메뉴"라는 표현은 이 가게가 한 가지 요리로 정해진 직관적이고도 간결한 운영 방식을 암시한다. 이는 주인공의 현재 삶이 복잡하기보다는 어떤 경계와 고정된 틀에 갇혀 있는 듯한 느낌을 강조한다.
  - 간판이 생긴 상황 자체가 주인공과 동생의 관계, 그리고 과거의 기억이 현재의 공간에 다시 자리 잡는 순간을 보여준다. 이미 다 끝난 일일지 모른다는 직감은, 사건의 흐름이 앞으로 어떻게 전개될지에 대한 암시를 남긴다.

- 근거가 된 episode: 1화 (1화. 간판이 하나 더 생겼다)
